In [2]:
#!/usr/bin/env python
"""
mysql_to_neo4j_improved.py
==========================

One-click migration of the **orders** MySQL schema → **Neo4j 5** database
`orders2`, incorporating **all requirements** discussed so far:

* dynamic schema discovery + surrogate-ID stripping
* 5 explicit join-tables converted to relationships (incl. dual mapping for
  `coupon_usages`)
* multi-threaded batch import (10 000 rows × 6 workers)
* idempotent MERGE, uniqueness constraints, rich logging
"""

from __future__ import annotations

import concurrent.futures
import json
import logging
import re
import sys
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from decimal import Decimal
from typing import Dict, List, Sequence, Tuple

import mysql.connector
import pymysql
from neo4j import GraphDatabase, Transaction

import time, functools
from neo4j.exceptions import TransientError

# --------------------------------------------------------------------------- #
# ░ CONFIGURATION ░
# --------------------------------------------------------------------------- #

# --- MySQL & Neo4j connection -----------------------------------------------
mysql.connector.connect = pymysql.connect  # PyMySQL drop-in

MYSQL = {
    "user": "dev",
    "password": "pwd",
    "host": "127.0.0.1",
    "database": "orders",
    "port": 3306,
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
}
NEO4J = {"uri": "bolt://localhost:7687", "user": "neo4j", "password": "gustjs21@"}
DB_NAME = "test01"  # Neo4j database

# --- Performance tuning -----------------------------------------------------
BATCH_SIZE = 10_000
MAX_WORKERS = 6

# --- Explicit join-table → relationship mappings ----------------------------
JOIN_TABLE_MAPPINGS: Dict[str, List[Dict[str, object]]] = {
    "menu_option_choice_to_menu_options": [
        {
            "rel_type": "HAS_CHOICE",
            "start_col": "menuOptionId",
            "end_col": "menuOptionChoiceId",
            "props": ["priority"],
        }
    ],
    "menu_option_to_menus": [
        {
            "rel_type": "HAS_OPTION",
            "start_col": "menuId",
            "end_col": "menuOptionId",
            "props": ["priority"],
        }
    ],
    "menu_to_inventory_items": [
        {
            "rel_type": "REQUIRES",
            "start_col": "menuId",
            "end_col": "inventoryItemId",
            "props": ["menuQuantityRequired"],
        }
    ],
    "refund_items": [
        {
            "rel_type": "REFUNDS_ITEM",
            "start_col": "refundId",
            "end_col": "orderItemId",
            "props": ["quantity"],
        }
    ],
    "coupon_usages": [
        {  # Customer ─ USED_COUPON → Coupon
            "rel_type": "USED_COUPON",
            "start_col": "customerId",
            "end_col": "couponId",
            "props": ["usedAt", "uuid", "createdBy"],
        },
        {  # Coupon ─ APPLIED_TO → Order
            "rel_type": "APPLIED_TO",
            "start_col": "couponId",
            "end_col": "orderId",
            "props": ["usedAt", "uuid", "createdBy"],
        },
    ],
}

JOIN_TABLE_MAPPINGS = {k.lower(): v for k, v in JOIN_TABLE_MAPPINGS.items()}

# --------------------------------------------------------------------------- #
# ░ LOGGING ░
# --------------------------------------------------------------------------- #
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(threadName)s ─ %(message)s",
    handlers=[logging.FileHandler("migration.log", "w"), logging.StreamHandler(sys.stdout)],
)

# --------------------------------------------------------------------------- #
# ░ DATA-CLASSES ░
# --------------------------------------------------------------------------- #


@dataclass
class ForeignKey:
    column: str
    ref_table: str
    ref_column: str


@dataclass
class TableMeta:
    name: str
    columns: Sequence[str]
    pk: Sequence[str]
    fks: Sequence[ForeignKey]
    is_join: bool = False
    extra_cols: Sequence[str] = field(default_factory=list)

    @property
    def business_keys(self) -> Sequence[str]:
        if self.pk and not self._is_surrogate_pk():
            return self.pk
        if self.is_join:
            return self.pk or [fk.column for fk in self.fks][:1]
        return self.pk or self.columns

    def _is_surrogate_pk(self) -> bool:
        return (
            len(self.pk) == 1
            and self.pk[0].lower() == "id"
            and (self.pk[0] not in self.columns or re.match(r"(int|bigint)", self.pk[0], re.I))
        )


# --------------------------------------------------------------------------- #
# ░  SCHEMA DISCOVERY  ░
# --------------------------------------------------------------------------- #

META_COLS = {"id", "createdAt", "updatedAt", "deletedAt", "created_at", "updated_at"}


class MySQLIntrospector:
    def __init__(self, mysql_cfg: Dict):
        self.conn = mysql.connector.connect(**mysql_cfg)
        self.schema = mysql_cfg["database"]

    def fetch_metadata(self) -> Dict[str, TableMeta]:
        cur = self.conn.cursor(pymysql.cursors.DictCursor)

        cur.execute(
            """SELECT TABLE_NAME, COLUMN_NAME
            FROM information_schema.columns
            WHERE table_schema = %s
                AND TABLE_NAME IN (
                    SELECT TABLE_NAME FROM information_schema.tables
                    WHERE table_schema = %s AND TABLE_TYPE = 'BASE TABLE'
                )""",
            (self.schema, self.schema),
        )

        col_map: Dict[str, List[str]] = defaultdict(list)
        for row in cur:
            col_map[row["TABLE_NAME"]].append(row["COLUMN_NAME"])

        cur.execute(
            """SELECT TABLE_NAME, COLUMN_NAME
               FROM information_schema.key_column_usage
               WHERE table_schema = %s AND constraint_name = 'PRIMARY'""",
            (self.schema,),
        )
        pk_map: Dict[str, List[str]] = defaultdict(list)
        for row in cur:
            pk_map[row["TABLE_NAME"]].append(row["COLUMN_NAME"])

        cur.execute(
            """SELECT k.TABLE_NAME, k.COLUMN_NAME,
                      k.REFERENCED_TABLE_NAME, k.REFERENCED_COLUMN_NAME
               FROM information_schema.key_column_usage k
               JOIN information_schema.table_constraints c
                 ON c.constraint_name = k.constraint_name
                AND c.table_name = k.table_name
               WHERE c.constraint_type = 'FOREIGN KEY'
                 AND k.table_schema = %s""",
            (self.schema,),
        )
        fk_map: Dict[str, List[ForeignKey]] = defaultdict(list)
        for row in cur:
            fk_map[row["TABLE_NAME"]].append(
                ForeignKey(
                    row["COLUMN_NAME"],
                    row["REFERENCED_TABLE_NAME"],
                    row["REFERENCED_COLUMN_NAME"],
                )
            )

        metadata: Dict[str, TableMeta] = {}
        for tbl, cols in col_map.items():
            pk = pk_map.get(tbl, [])
            fks = fk_map.get(tbl, [])
            is_join = self._detect_join_table(tbl, cols, pk, fks)
            extras = [c for c in cols if c not in pk and c not in [fk.column for fk in fks]]
            metadata[tbl] = TableMeta(tbl, cols, pk, fks, is_join, extras)
        cur.close()
        return metadata

    @staticmethod

    def _detect_join_table(
        tbl_name: str,
        columns: Sequence[str],
        pk: Sequence[str],
        fks: Sequence[ForeignKey],
    ) -> bool:
        tbl_key = tbl_name.strip("` ").lower() # ← 소문자 변환 + 백틱 제거

        if tbl_key in JOIN_TABLE_MAPPINGS:     # ← 여기도 tbl_key 사용
            return True

        if len(fks) < 2:
            return False

        fk_cols = {fk.column for fk in fks}
        non_meta = set(columns) - META_COLS
        return non_meta.issubset(fk_cols | set(pk))

# --------------------------------------------------------------------------- #
# ░ MIGRATOR ░
# --------------------------------------------------------------------------- #

# ─── neo_utils.py ─────────────────────────────────────────────────
def run_tx_with_retry(session, query, **params):
    MAX_RETRY = 5
    for attempt in range(1, MAX_RETRY + 1):
        try:
            return session.run(query, **params)
        except TransientError as e:
            if "DeadlockDetected" not in e.code:
                raise
            backoff = 0.3 * (2 ** (attempt - 1))
            logging.warning("Dead-lock detected → retry %d/5 in %.1fs", attempt, backoff)
            time.sleep(backoff)
    raise RuntimeError("Aborted after 5 retries (dead-lock)")

class Migrator:
    def __init__(self, mysql_cfg: Dict, neo_cfg: Dict[str, str]):
        logging.info("Connecting to databases…")
        self.mysql_cfg = mysql_cfg
        self.neo4j = GraphDatabase.driver(neo_cfg["uri"], auth=(neo_cfg["user"], neo_cfg["password"]))
        self.meta = MySQLIntrospector(mysql_cfg).fetch_metadata()

    # ------------------------------------------------------------------- #
    # public
    # ------------------------------------------------------------------- #
    def run(self) -> None:
        self._clear_neo4j()
        self._create_constraints()

        entities = [m for m in self.meta.values() if not m.is_join]
        joins    = [m for m in self.meta.values() if m.is_join]

        logging.info(">> NODE phase (%d tables)…", len(entities))
        self._parallel(entities, self._migrate_entity)

        logging.info(">> JOIN-TABLE → REL phase (%d)…", len(joins))
        self._parallel(joins, self._migrate_join_table)

        logging.info(">> FK-REL phase (all remaining FK edges)…")
        self._migrate_all_fk_relationships(entities)

        logging.info(">> Removing surrogate IDs from nodes…")
        self._remove_surrogate_ids()

        logging.info("[OK] Migration finished successfully.")


    def close(self) -> None:
        self.neo4j.close()

    # ------------------------------------------------------------------- #
    # internal helpers
    # ------------------------------------------------------------------- #
    def _parallel(self, metas: Sequence[TableMeta], fn):
        with concurrent.futures.ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(fn, m): m.name for m in metas}
            for fut in concurrent.futures.as_completed(futs):
                tbl = futs[fut]
                try:
                    fut.result()
                    logging.info("Table '%s' done.", tbl)
                except Exception as exc:
                    logging.exception("Table '%s' failed - %s", tbl, exc)


    # ---------- entity nodes ------------------------------------------- #
    def _migrate_entity(self, meta: TableMeta) -> None:
        conn = mysql.connector.connect(**self.mysql_cfg)
        cur = conn.cursor(pymysql.cursors.DictCursor)

        label = self._to_label(meta.name)
        offset = 0
        where_not_deleted = " WHERE deletedAt IS NULL" if "deletedAt" in meta.columns else ""
        while True:
            cur.execute(f"SELECT * FROM `{meta.name}`{where_not_deleted} LIMIT {BATCH_SIZE} OFFSET {offset}")
            rows = cur.fetchall()
            if not rows:
                break
            cleaned = [self._clean(row) for row in rows]
            self._write_nodes(label, meta.business_keys, cleaned, meta._is_surrogate_pk())
            offset += BATCH_SIZE

        cur.close()
        conn.close()

    def _write_nodes(self, label: str, keys: Sequence[str], rows: List[Dict], has_surrogate: bool):
        rows = [r for r in rows if all(r.get(k) is not None for k in keys)]
        if not rows:
            return
        merge_on = ", ".join(f"{k}: row.{k}" for k in keys)
        query = f"""
        UNWIND $rows AS row
        MERGE (n:{label} {{ {merge_on} }})
        SET n += row
        """
        if has_surrogate:
            query += "\nREMOVE n.id"
        with self.neo4j.session(database=DB_NAME) as sess:
            run_tx_with_retry(sess, query, rows=rows)

    # ---------- join-tables → relationships ---------------------------- #
    def _migrate_join_table(self, meta: TableMeta) -> None:
        tbl_key = meta.name.strip("` ").lower()          # ← 정규화

        mappings = JOIN_TABLE_MAPPINGS.get(
            tbl_key,
            [  # fallback: first two FKs
                {
                    "rel_type": self._to_rel_type(meta.name),
                    "start_col": meta.fks[0].column,
                    "end_col":  meta.fks[1].column,
                    "props":    meta.extra_cols,
                }
            ],
        )

        for mapping in mappings:
            self._process_mapping(meta, mapping)


    def _process_mapping(self, meta: TableMeta, mapping: Dict[str, object]) -> None:
        start_fk = self._find_fk(meta, mapping["start_col"])
        end_fk = self._find_fk(meta, mapping["end_col"])
        rel_type: str = mapping["rel_type"]  # type: ignore
        prop_cols: List[str] = mapping["props"] if mapping.get("props") else meta.extra_cols  # type: ignore

        conn = mysql.connector.connect(**self.mysql_cfg)
        cur = conn.cursor(pymysql.cursors.DictCursor)
        offset = 0
        while True:
            cur.execute(f"SELECT * FROM `{meta.name}` LIMIT {BATCH_SIZE} OFFSET {offset}")
            rows = cur.fetchall()
            if not rows:
                break
            cleaned = [self._clean(row, keep_cols=[start_fk.column, end_fk.column] + prop_cols) for row in rows]
            self._write_relationships(rel_type, start_fk, end_fk, cleaned, prop_cols)
            offset += BATCH_SIZE
        cur.close()
        conn.close()

    def _write_relationships(
        self,
        rel_type: str,
        start_fk: ForeignKey,
        end_fk: ForeignKey,
        rows: List[Dict],
        prop_cols: Sequence[str],
    ):
        if not rows:
            return
        prop_set   = ", ".join(f'r.{c}=row.{c}' for c in prop_cols) if prop_cols else ""
        cypher_body = f"""
        MATCH (a:{self._to_label(start_fk.ref_table)} {{ {start_fk.ref_column}: row.{start_fk.column} }})
        MATCH (b:{self._to_label(end_fk.ref_table)}   {{ {end_fk.ref_column}: row.{end_fk.column} }})
        MERGE (a)-[r:{rel_type}]->(b)
        {"SET " + prop_set if prop_set else ""}
        """

        apoc_call = """
        CALL apoc.periodic.iterate(
          'UNWIND $rows AS row RETURN row',
          $cypher,
          {batchSize:1000, parallel:true, params:{rows:$rows}}
        )
        """
        with self.neo4j.session(database=DB_NAME) as sess:
            sess.run(apoc_call, parameters={"rows": rows, "cypher": cypher_body})
    
        # ---------- FK (entity → parent) relationships -------------------- #
    def _migrate_all_fk_relationships(self, entities: Sequence[TableMeta]) -> None:
        """Create relationships for every foreign-key that is *not* a join table."""
        # (meta, fk) 튜플 목록 만들기
        fk_tasks: List[Tuple[TableMeta, ForeignKey]] = []
        for child in entities:
            for fk in child.fks:
                parent_meta = self.meta[fk.ref_table]
                if parent_meta.is_join:      # 부모가 조인테이블이면 skip
                    continue
                fk_tasks.append((child, fk))

        # 병렬 실행
        with concurrent.futures.ThreadPoolExecutor(MAX_WORKERS) as ex:
            futs = {ex.submit(self._process_fk_pair, c, fk): (c.name, fk.column) for c, fk in fk_tasks}
            for fut in concurrent.futures.as_completed(futs):
                child, col = futs[fut]
                try:
                    fut.result()
                    logging.info("FK-REL done: %s.%s", child, col)
                except Exception as exc:
                    logging.exception("FK-REL FAILED: %s.%s - %s", child, col, exc)

    def _process_fk_pair(self, child: TableMeta, fk: ForeignKey) -> None:
        """Create one FK edge type for (child_table.column → parent_table)."""
        child_label  = self._to_label(child.name)
        parent_label = self._to_label(fk.ref_table)

        # 관계 타입: CHILDLABEL_TO_PARENTLABEL  (예: Orderitem_TO_Menu)
        rel_type = f"{child_label.upper()}_TO_{parent_label.upper()}"

        # 쿼리 준비 ----------------------------------------------------------------
        child_keys = child.business_keys
        match_child = ", ".join(f"{k}: row.{k}" for k in child_keys)

        worker_cypher = f"""
        MATCH (c:{child_label}  {{ {match_child} }})
        MATCH (p:{parent_label} {{ {fk.ref_column}: row.{fk.column} }})
        MERGE (c)-[r:{rel_type}]->(p)
        """

        # 데이터 추출 --------------------------------------------------------------
        cols_needed = list(child_keys) + [fk.column]
        select_cols = ", ".join(f"`{c}`" for c in cols_needed)

        conn = mysql.connector.connect(**self.mysql_cfg)
        cur  = conn.cursor(pymysql.cursors.DictCursor)
        offset = 0

        while True:
            cur.execute(
                f"SELECT {select_cols} FROM `{child.name}` LIMIT {BATCH_SIZE} OFFSET {offset}"
            )
            rows = cur.fetchall()
            if not rows:
                break
            cleaned = [self._clean(row, keep_cols=cols_needed) for row in rows]

            apoc_call = f"""
            CALL apoc.periodic.iterate(
              'UNWIND $rows AS row RETURN row',
              '{worker_cypher}',
              {{batchSize:1000, parallel:true, params:{{rows:$rows}}}}
            )
            """
            with self.neo4j.session(database=DB_NAME) as sess:
                sess.run(apoc_call, parameters={"rows": cleaned})
            offset += BATCH_SIZE
        cur.close()
        conn.close()

    # ---------- Neo4j maintenance -------------------------------------- #
    def _clear_neo4j(self):
        logging.warning("Clearing Neo4j database '%s'…", DB_NAME)
        with self.neo4j.session(database=DB_NAME) as sess:
            sess.run("MATCH (n) DETACH DELETE n")

    def _create_constraints(self):
        logging.info("Creating uniqueness constraints…")
        with self.neo4j.session(database=DB_NAME) as sess:
            for meta in self.meta.values():
                if meta.is_join: # ㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠㅠ
                    continue
                label = self._to_label(meta.name)
                for key in meta.business_keys:
                    sess.run(f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.{key} IS UNIQUE")

    def _remove_surrogate_ids(self):
        with self.neo4j.session(database=DB_NAME) as sess:
            for meta in self.meta.values():
                if meta._is_surrogate_pk():
                    sess.run(f"MATCH (n:{self._to_label(meta.name)}) WHERE exists(n.id) REMOVE n.id")

    # ---------- utility ------------------------------------------------ #
    @staticmethod
    def _find_fk(meta: TableMeta, col_name: str) -> ForeignKey:
        for fk in meta.fks:
            if fk.column == col_name:
                return fk
        raise ValueError(f"FK column '{col_name}' not found in table '{meta.name}'")

    @staticmethod
    def _to_label(table_name: str) -> str:
        return "".join(x.capitalize() for x in re.split(r"[_\W]+", table_name))

    @staticmethod
    def _to_rel_type(table_name: str) -> str:
        return re.sub(r"[_\W]+", "_", table_name).upper()

    @staticmethod
    def _clean(row: Dict, keep_cols: Sequence[str] | None = None) -> Dict:
        """
        • drop NULLs & non-essential timestamps
        • serialize datetimes / decimals
        • optionally keep only specified columns (join-tables)
        """
        cleaned = {}
        for col, val in row.items():
            if keep_cols and col not in keep_cols:
                continue
            if val is None or (
                isinstance(val, datetime)
                and col.lower() in {"createdat", "updatedat", "deletedat", "created_at", "updated_at"}
            ):
                continue
            if isinstance(val, datetime):
                cleaned[col] = val.isoformat()
            elif isinstance(val, Decimal):
                cleaned[col] = float(val)
            elif isinstance(val, (dict, list)):
                cleaned[col] = json.dumps(val)
            else:
                cleaned[col] = val
        return cleaned

# --------------------------------------------------------------------------- #
# ░ MAIN ░
# --------------------------------------------------------------------------- #
def main():
    migrator = Migrator(MYSQL, NEO4J)
    try:
        migrator.run()
    finally:
        migrator.close()


if __name__ == "__main__":
    main()

2025-07-07 14:49:23,203 [INFO] MainThread ─ Connecting to databases…
2025-07-07 14:49:23,299 [WARNING] MainThread ─ Clearing Neo4j database 'test01'…
2025-07-07 14:49:25,575 [INFO] MainThread ─ Creating uniqueness constraints…
2025-07-07 14:49:28,886 [INFO] MainThread ─ >> NODE phase (23 tables)…
2025-07-07 14:49:30,243 [INFO] MainThread ─ Table 'countries' done.
2025-07-07 14:49:30,536 [INFO] MainThread ─ Table 'inventory_items' done.
2025-07-07 14:49:31,041 [INFO] MainThread ─ Table 'inventory_transactions' done.
2025-07-07 14:49:31,886 [INFO] MainThread ─ Table 'food_courts' done.
2025-07-07 14:49:32,520 [INFO] MainThread ─ Table 'coupons' done.
2025-07-07 14:49:32,692 [INFO] MainThread ─ Table 'categories' done.
2025-07-07 14:49:32,878 [INFO] MainThread ─ Table 'locales' done.
2025-07-07 14:49:33,037 [INFO] MainThread ─ Table 'grouped_texts' done.
2025-07-07 14:49:34,225 [INFO] MainThread ─ Table 'menu_options' done.
2025-07-07 14:49:34,491 [INFO] MainThread ─ Table 'payment_option